## Conduct Deep Research 
<p>for a given topic and send email and push notification </p>

In [ ]:
#use mailtrap as the mail client
!uv pip install mailtrap serpapi

In [ ]:
from agents import Agent, trace, Runner, function_tool
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import asyncio
import mailtrap
import os
import requests
from IPython.display import display, Markdown
import serpapi
from datetime import datetime
import gradio as gr

In [ ]:
load_dotenv(override=True)
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
openai_api_key = os.getenv("OPENAI_API_KEY")
serp_api_key = os.getenv("SERP_API_KEY")
mailtrap_api_key = os.getenv("MAILTRAP_API_KEY")
sender_email = os.getenv("EMAIL_SENDER")
receiver_email = os.getenv("EMAIL_RECEIVER")

if not openai_api_key:
    raise ValueError("OPENAI_API_KEY is not set")

if pushover_user is None or pushover_token is None:
    raise ValueError("Pushover credentials are required!")

if serp_api_key is None:
    raise ValueError("Serp API key is required for Google search")

if mailtrap_api_key is None:
    raise ValueError("Mailtrap API key is required for email sending")

if sender_email is None or receiver_email is None:
    raise ValueError("Email sender and receiver are required")



In [ ]:

@function_tool
def send_email(subject: str, html_body: str):
    """Sends the email with the given subject and HTML body."""
    try:
        client = mailtrap.MailtrapClient(token=mailtrap_api_key)
        mail = mailtrap.Mail(
            sender=mailtrap.Address(email=sender_email, name="Dear Manager"),
            to=[mailtrap.Address(email=receiver_email)],
            subject=subject,
            html=html_body,
        )
        client.send(mail)
        return "Email sent successfully."
    except Exception as e:
        return f"Failed to send email: {e}"

In [ ]:
subject_instructions = "You can write a subject for a topic email. \
You are given a message and you need to write a subject for an email that is likely to get a response."

html_instructions = "You can convert a text email body to an HTML email body. \
You are given a text email body which might have some markdown \
and you need to convert it to an HTML email body with simple, clear, compelling layout and design."

subject_writer = Agent(name="Email subject writer", instructions=subject_instructions, model="gpt-4o-mini")
subject_tool = subject_writer.as_tool(tool_name="subject_writer", tool_description="Write a subject for a research topic")

html_converter = Agent(name="HTML email body converter", instructions=html_instructions, model="gpt-4o-mini")
html_tool = html_converter.as_tool(tool_name="html_converter",tool_description="Convert a text email body to an HTML email body")

In [ ]:
@function_tool
def send_push_notification(message: str):
    """Sends a push notification to the user on research complete."""
    try:
        url = "https://api.pushover.net/1/messages.json"
        payload = {
            "token": pushover_token,
            "user": pushover_user,
            "message": message
        }
        result = requests.post(url, payload)
        result.raise_for_status()
        return "Push notification sent."
    except Exception as e:
        return f"Push notification failed: {e}"

In [ ]:
email_tools = [subject_tool, html_tool, send_push_notification, send_email]

In [ ]:
instructions ="You are an email formatter and sender. You receive the body of an email to be sent. \
You first use the subject_writer tool to write a subject for the email, then use the html_converter tool to convert the body to HTML. \
addtionally send a push notification with the send_push_notification tool to inform the user of research completion. Finally, you use the send_email tool to send the email with the subject and HTML body."


emailer_agent = Agent(
    name="Email Manager",
    instructions=instructions,
    tools=email_tools,
    model="gpt-4o-mini",
    handoff_description="Create a subject, format email to html and send it")

In [ ]:
@function_tool
def google_search(query: str):
    """Searches Google for information on a given query."""
    try:
        client = serpapi.Client(api_key=serp_api_key)
        response = client.search(engine="google", q=query)
        return response["organic_results"]
    except Exception as e:
        return {"error": str(e), "organic_results": []}

In [ ]:
HOW_MANY_SEARCHES = 5

planner_instruction = f"You are a helpful research assistant. Given a query, come up with a set of web searches \
to perform to best answer the query. Output {HOW_MANY_SEARCHES} terms to query for."

class WebSearchItem(BaseModel):
    reason: str = Field(description="Your reasoning for why this search is important to the query.")

    query: str = Field(description="The search term to use for the web search.")


class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="A list of web searches to perform to best answer the query.")


planner_agent = Agent(
    name="PlannerAgent",
    instructions=planner_instruction,
    model="gpt-4o-mini",
    output_type=WebSearchPlan,
)

planner_agent_tool = planner_agent.as_tool(tool_name="planner_agent", tool_description="Plan a web search")

In [ ]:
research_tools = [planner_agent_tool, google_search]

In [ ]:

current_date = datetime.now().strftime("%Y-%m")

In [ ]:
report_writer_instruction = f"You are a senior researcher tasked with writing a cohesive report for a research query.\
    You will be provided with the original query, use the planner_agent tool to plan the initial research terms to.\
    Using the google_search tool, you will then search the web for upto date {current_date} information on each of the items provided by the planner_agent. \
    Come up with an outline for the report that describes the structure and \
    flow of the report. Then, generate the report and return that as your final output.\
    The final output should be in markdown format, and it should be lengthy and detailed. Aim \
    for 5-10 pages of content, at least 1000 words. hand off to the emailer_agent to send the report as an email."


class ReportData(BaseModel):
    short_summary: str = Field(description="A short 2-3 sentence summary of the findings.")

    markdown_report: str = Field(description="The final report")

    follow_up_questions: list[str] = Field(description="Suggested topics to research further")


writer_agent = Agent(
    name="WriterAgent",
    instructions=report_writer_instruction,
    model="gpt-4o-mini",
    tools=research_tools,
    handoffs=[emailer_agent],
    output_type=ReportData,
)

In [ ]:
async def run_research(query: str):
    yield "**Starting research...**\n\nPlanning search queries and gathering sources.\n"
    try:
        with trace("Research Workflow"):
            result = await Runner.run(writer_agent, query, max_turns=10)
        yield "**Report generated.**\n\nFormatting and sending email...\n"
        if hasattr(result, "markdown_report"):
            yield result.markdown_report
        elif isinstance(result, str):
            yield result
        else:
            yield str(result) if result is not None else "Research completed (no report content)."
    except Exception as e:
        yield f"** Error**\n\n{e}"

In [ ]:
async def run(query: str):
    async for chunk in run_research(query):
        yield chunk


with gr.Blocks(theme=gr.themes.Default(primary_hue="sky")) as ui:
    gr.Markdown("# Deep Research")
    query_textbox = gr.Textbox(label="What topic would you like to research?")
    run_button = gr.Button("Run", variant="primary")
    report = gr.Markdown(label="Report")
    
    run_button.click(fn=run, inputs=query_textbox, outputs=report)
    query_textbox.submit(fn=run, inputs=query_textbox, outputs=report)

ui.launch(inbrowser=True)